[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Guía del tema 02](README.md)

# Sincronización, condición y deadlock

**Tema:** 02 · **Sesiones:** 8, 10 · **Edición:** 1.0.2026

**Pregunta guía:** ¿Qué invariante protege cada primitiva y cómo se detecta un ciclo de espera?


## Cómo usar este notebook

Sigue la secuencia sin saltar directamente al código:

1. Comprueba los prerrequisitos y formula una respuesta inicial a la pregunta guía.
2. Estudia la explicación paso a paso y reconstruye el mapa visual.
3. Predice el resultado del ejemplo resuelto antes de ejecutar su celda.
4. Repite el razonamiento en el ejemplo guiado y contesta las preguntas de comprensión.
5. Solo entonces desarrolla los ejercicios progresivos y contrasta los criterios de aceptación.

La meta no es memorizar una salida: es poder explicar qué se calculó, bajo qué supuestos y con qué evidencia.


## Antes de empezar

**Por qué importa.** La sincronización protege invariantes y establece visibilidad; no es un adorno que se agrega después de encontrar una carrera.

**Prerrequisitos.**

- Partición de datos y referencia serial del tema 01.
- Punteros, funciones y vida útil de objetos en C/C++.

**Diagnóstico inicial.** Escribe una respuesta de dos frases a la pregunta guía. Al terminar, vuelve a leerla y señala qué corregiste.


## Resultados de aprendizaje

Al finalizar podrás:

- Relacionar mutex, condición y barrera con invariantes.
- Explicar happens-before sin usar tiempo como sincronización.
- Detectar un ciclo en un grafo wait-for.


## Explicación paso a paso

En esta sección todavía no se busca programar. Primero se construye el modelo mental que permitirá leer el ejemplo y detectar conclusiones inválidas.

### Paso 1: construye la idea

Un mutex protege un invariante, no una línea aislada.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Paso 2: construye la idea

Una variable de condición se espera dentro de un bucle que reevalúa el predicado.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Paso 3: construye la idea

Un deadlock requiere exclusión, retención y espera, no expropiación y espera circular.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Vocabulario mínimo

- carrera — accesos concurrentes incompatibles sin orden suficiente
- happens-before — relación que hace visible un efecto entre hilos
- deadlock — ciclo de espera que impide el progreso


## Mapa visual

Los diagramas se almacenan en la carpeta compartida [`curso/images/`](../../images/README.md). Úsalos para explicar relaciones y secuencias; no los trates como resultados experimentales.

### Happens Before

![Publicación de datos entre productor y consumidor](../../images/happens-before.svg)

**Cómo leerlo.** La flecha central representa sincronización, no el mero paso del tiempo. Sin esa relación el consumidor no tiene garantía de observar la escritura.


In [ ]:
from pathlib import Path

def find_repository(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "INDICE_CURSO.md").is_file():
            return candidate
    raise RuntimeError("No se encontró la raíz del repositorio")

ROOT = find_repository(Path.cwd())
TOPIC = "02"
NOTEBOOK = "02_memoria_compartida/02_sincronizacion.ipynb"
assert (ROOT / "curso" / "notebooks" / "02_memoria_compartida" / "README.md").is_file()
print(f"Repositorio: {ROOT}")
print(f"Notebook: {NOTEBOOK}")


## Ejemplo resuelto: Grafo de espera

**Situación.** Se detecta un ciclo mediante búsqueda en profundidad.

**Razonamiento antes del código.**

1. Identifica entradas, supuestos y la magnitud que debe producirse.
2. Formula una propiedad esperada; la celda la expresa mediante una aserción.
3. Predice el resultado y después ejecuta. Una salida impresa ayuda a observar, pero la aserción decide si se conserva el invariante.


In [ ]:
def has_cycle(graph):
    visiting, done = set(), set()
    def visit(node):
        if node in visiting: return True
        if node in done: return False
        visiting.add(node)
        if any(visit(next_node) for next_node in graph.get(node, ())): return True
        visiting.remove(node); done.add(node); return False
    return any(visit(node) for node in graph)
safe = {"T0": ["T1"], "T1": []}
deadlock = {"T0": ["T1"], "T1": ["T0"]}
assert not has_cycle(safe) and has_cycle(deadlock)
print("seguro:", has_cycle(safe), "deadlock:", has_cycle(deadlock))


### Explicación del resultado

Ordenar globalmente la adquisición de recursos rompe la condición de espera circular.

**Qué debes poder explicar.** Relaciona cada valor producido con el modelo conceptual y distingue el cálculo ilustrativo de una medición sobre hardware real.


## Ejemplo guiado: Buffer acotado

**Situación.** Se comprueban invariantes de ocupación para una secuencia productor-consumidor.

**Tu turno antes de ejecutar.** Anota una predicción, identifica la variable que modificarías y explica qué propiedad no debe cambiar. Ejecuta después y compara el resultado con tu predicción.


In [ ]:
from collections import deque
capacity, buffer = 3, deque()
operations = [("put", 4), ("put", 7), ("get", None), ("put", 9), ("get", None), ("get", None)]
consumed = []
for operation, value in operations:
    if operation == "put":
        assert len(buffer) < capacity
        buffer.append(value)
    else:
        assert buffer
        consumed.append(buffer.popleft())
    assert 0 <= len(buffer) <= capacity
assert consumed == [4, 7, 9]
print(consumed)


### Lectura razonada

En Pthreads, el predicado sería `count>0` o `count<capacity` protegido por el mismo mutex.

**Qué debes poder explicar.** Relaciona cada valor producido con el modelo conceptual y distingue el cálculo ilustrativo de una medición sobre hardware real.


## Comprueba tu comprensión

1. ¿Por qué una variable condición debe comprobarse dentro de un bucle y bajo el mismo mutex que protege el estado?
2. ¿Qué aserción o comparación del ejemplo protege la corrección y qué error detectaría?
3. ¿Qué parte es un modelo y qué evidencia adicional exigirías antes de generalizar al hardware real?

Responde primero sin ejecutar código. Luego usa las celdas anteriores para corregir o precisar tu explicación.


## Ejercicios progresivos

### Nivel 1 — reproducir y explicar

Cambia un parámetro del ejemplo resuelto, predice el efecto y explica por qué la aserción debe seguir pasando o debe fallar de manera controlada.

### Nivel 2 — aplicar

1. Anotar el invariante al lado de cada estado compartido.
2. Construir un caso que fuerce intercalaciones distintas.
3. Ejecutar ThreadSanitizer cuando el toolchain lo soporte.

### Nivel 3 — producir evidencia

Conserva entrada, comandos, versión del entorno, resultados crudos y una conclusión limitada por los supuestos. Separa siempre corrección, tiempo de kernel y tiempo extremo a extremo cuando corresponda.


## Errores frecuentes

- Usar `sleep` para ordenar hilos.
- Esperar condición con `if` en lugar de `while`.
- Bloquear recursos en órdenes diferentes.


## Criterios de aceptación

- Invariantes escritos y comprobados.
- Ausencia de ciclos en el orden de locks.
- Pruebas repetidas y detector de carreras documentado.


## Síntesis

- La pregunta que debes poder responder es: **¿Qué invariante protege cada primitiva y cómo se detecta un ciclo de espera?**
- Los ejemplos convierten el modelo en propiedades comprobables; no sustituyen una medición del sistema objetivo.
- Los ejercicios se consideran terminados cuando la explicación, la corrección y la evidencia satisfacen los criterios de aceptación.


## Referencias y material relacionado

- [Mutex](../../../pthreads/thread_mutex.c)
- [Deadlock](../../../pthreads/thread_deadlock.c)


[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Continuar desde la guía del tema 02](README.md)
